In [ ]:
# Célula 1: Instalações
# 'transformers[torch]' instala o transformers e o PyTorch
# 'datasets' é a biblioteca da Hugging Face para carregar dados
# 'scikit-learn' é para calcular nossa métrica de acurácia (MAE)
!pip install transformers[torch] datasets scikit-learn

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# --- 1. CARREGAR OS DADOS ---
#
# INSTRUÇÃO:
# 1. No painel à esquerda no Colab, clique no ícone de 'Pasta'.
# 2. Clique no ícone 'Fazer upload' (folha com seta para cima).
# 3. Selecione o seu arquivo .csv unido (que você exportou do Power BI).
#
# Substitua 'seu_arquivo_unido.csv' pelo nome EXATO do arquivo que você enviou.
file_name = '/content/sample_data/data_final.csv'

try:
    df = pd.read_csv(file_name)
    print("Arquivo CSV carregado com sucesso!")
    print(df.head())
except FileNotFoundError:
    print(f"--- ERRO ---")
    print(f"O arquivo '{file_name}' não foi encontrado.")
    print("Por favor, verifique se o nome está correto e se você fez o upload.")


# --- 2. DEFINIR FEATURES (X) E TARGET (y) ---
#
# Com base nas suas capturas de tela, assumi que:
# X (Features) = A coluna 'essay' (o texto da redação)
# y (Target)   = A coluna 'final_grade' (a nota que você quer prever)
#
# Altere os nomes das colunas aqui se forem diferentes.
feature_column = 'essay'
target_column = 'Soma de final_grade'

if feature_column in df.columns and target_column in df.columns:
    X = df[feature_column]
    y = df[target_column]

    # --- 3. DIVIDIR OS DADOS (SPLIT) ---
    #
    # Vamos dividir em 80% para treino e 20% para teste.
    # O 'random_state=42' é uma convenção para garantir que a divisão
    # seja sempre a mesma, tornando seu experimento reprodutível.

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,       # 20% dos dados serão usados para teste
        random_state=42      # Para reprodutibilidade
    )

    # --- 4. VERIFICAR OS RESULTADOS ---
    print("\n--- Divisão Concluída ---")
    print(f"Total de Amostras: {len(df)}")
    print(f"Amostras de Treino (X_train): {len(X_train)}")
    print(f"Amostras de Teste (X_test): {len(X_test)}")

    # --- 5. SALVAR OS NOVOS ARQUIVOS ---
    #
    # Vamos juntar X e y novamente nos seus respectivos arquivos
    # para facilitar o carregamento no seu script de treino.

    train_df = pd.DataFrame({feature_column: X_train, target_column: y_train})
    test_df = pd.DataFrame({feature_column: X_test, target_column: y_test})

    # Salvar em novos arquivos CSV
    train_df.to_csv('train_data.csv', index=False)
    test_df.to_csv('test_data.csv', index=False)

    print("\nArquivos 'train_data.csv' e 'test_data.csv' salvos no ambiente do Colab.")
    print("Você pode baixá-los no painel de 'Arquivos' à esquerda.")

else:
    print(f"--- ERRO ---")
    print(f"As colunas '{feature_column}' ou '{target_column}' não foram encontradas no CSV.")
    print(f"Colunas disponíveis: {df.columns.tolist()}")

In [ ]:
# @title
# Célula 2 (VERSÃO ATUALIZADA - Treinando com o Tema)
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

# 1. Escolha do Modelo (BERTimbau)
model_checkpoint = "neuralmind/bert-large-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 2. Define os arquivos que vamos usar
# --- MUDANÇA IMPORTANTE 1: Usar os novos arquivos que criámos ---
data_files = {
    "train": "/content/sample_data/train_dataset.csv",
    "test": "/content/sample_data/test_dataset.csv"
}

# 3. Carrega os CSVs
# (O delimiter=';' está correto, pois salvámos nesse formato)
datasets = load_dataset('csv', data_files=data_files, delimiter=';')
print("Dados (com tema) carregados:")
print(datasets)

# 4. Função para formatar as notas (labels)
# (Esta função está correta e não muda)
# Normalizamos a nota [0, 1000] para o intervalo [0, 1]
def formatar_para_nota_final(example):
    example['labels'] = float(example['nota_final'] / 1000.0)
    return example

# 5. Função de tokenização (com o TEMA)
# --- MUDANÇA IMPORTANTE 2: Nova função de tokenização ---
def tokenize_com_tema(examples):
    # O tokenizer do BERT entende dois segmentos de texto.
    # Ele vai formatar o input como: [CLS] tema [SEP] texto_original [SEP]
    return tokenizer(
        examples["tema_redacao"],    # Segmento A
        examples["texto_original"], # Segmento B
        padding="max_length",
        truncation=True,
        max_length=512 # O max_length total para ambos os segmentos
    )

# 6. Aplicar tudo nos datasets
print("Formatando e Normalizando as notas (labels)...")
formatted_datasets = datasets.map(formatar_para_nota_final)

print("Tokenizando os textos (com o Tema)...")
# Aplicar a nova função de tokenização
tokenized_datasets = formatted_datasets.map(tokenize_com_tema, batched=True)

# 7. Limpeza final
# --- MUDANÇA IMPORTANTE 3: Remover as colunas de texto originais ---
tokenized_datasets = tokenized_datasets.remove_columns([
    "tema_redacao",       # Remover
    "texto_original",     # Remover
    "nota_final"     # Remover
])

print("\nDatasets prontos para o treino (com Tema):")
print(tokenized_datasets["train"])
print("\nExemplo de 'label' (nota) agora está entre 0 e 1:")
print(tokenized_datasets["train"][0]['labels'])

In [ ]:
# Célula 3 (Corrigida com Normalização)
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import mean_absolute_error

# 1. Carregar o Modelo para REGRESSÃO (prever 1 número)
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=1
)

# 2. Definir a Métrica (MAE)
# --- MUDANÇA IMPORTANTE AQUI ---
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Inverte a normalização para que o erro seja em pontos (0-1000)
    predictions_scaled = predictions.flatten() * 1000.0
    labels_scaled = labels * 1000.0

    mae = mean_absolute_error(labels_scaled, predictions_scaled)
    return {"mean_absolute_error": mae}

print("Modelo e métricas (com escala) prontos.")

In [ ]:
from transformers import TrainingArguments, Trainer

# 1. Argumentos do Treino (FOCO: PRECISÃO MÁXIMA)
training_args = TrainingArguments(
    # --- MUDANÇA 1: O NOME (refletindo o treino com TEMA) ---
    output_dir="modelo_nota_final_bert_large_COM_TEMA_v1",

    # --- OTIMIZAÇÕES T4 (Estabilidade - Manter) ---
    fp16=True,                          # Ativa o Mixed Precision (essencial no T4)
    per_device_train_batch_size=8,      # Lote real
    gradient_accumulation_steps=4,      # Lote Efetivo = 32

    # --- MUDANÇA 2: FOCO EM PRECISÃO (Mais tempo e melhor scheduler) ---
    learning_rate=1e-5,                 # Manter o LR (é um valor seguro)
    num_train_epochs=8,                 # <--- Aumentado de 5 para 8
    lr_scheduler_type="cosine",         # <--- Mudar de 'linear' (padrão) para 'cosine'

    # --- Configs Padrão (Manter) ---
    per_device_eval_batch_size=16,      # Pode aumentar na avaliação
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,        # O mais importante! Salva o melhor modelo.
    report_to="none",
)

# 2. Criar o 'Trainer'
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
    # callbacks=[EarlyStopping(patience=2)]
)

print("Trainer configurado para PRECISÃO MÁXIMA (Com Tema, 8 Épocas, Scheduler 'cosine').")
print("Pronto para treinar (Célula 5).")

In [ ]:
# Célula 5: Treinar o modelo
print("Iniciando o treinamento do modelo de NOTA FINAL...")

trainer.train()

print("Treinamento concluído!")

In [ ]:
# Célula 6: Avaliação Final
print("Avaliando o modelo no dataset de teste (dados nunca vistos)...")

resultados = trainer.evaluate()

print("--- RESULTADO FINAL (Nota Final 0-1000) ---")
print(f"Erro Médio (MAE): {resultados['eval_mean_absolute_error']:.2f} pontos")
print("---------------------------------------------")

In [ ]:
# Célula para Salvar o Modelo no Google Drive

from google.colab import drive
import shutil

# 1. Montar o seu Google Drive
print("A pedir permissão para aceder ao Google Drive...")
drive.mount('/content/drive')
print("Google Drive montado com sucesso!")

# 2. Definir os caminhos

pasta_modelo_no_colab = "modelo_nota_final_bert_large_v_melhorado" # Verifique se este é o nome certo
pasta_destino_no_drive = "/content/drive/MyDrive/meu_modelo_bert_redacoes"

# 3. Copiar os ficheiros
print(f"A copiar os ficheiros do modelo '{pasta_modelo_no_colab}' para o seu Google Drive...")
shutil.copytree(pasta_modelo_no_colab, pasta_destino_no_drive)

print("\n--- SUCESSO! ---")
print(f"O seu modelo foi salvo com sucesso na pasta '{pasta_destino_no_drive}' do seu Google Drive.")

In [ ]:
# Célula de Diagnóstico (Nota Final)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Analisando a distribuição das notas em TREINO (/content/sample_data/train_dataset_final.csv):")
df_train = pd.read_csv('/content/sample_data/train_dataset_final.csv', delimiter=';')

# 1. Vamos ver o resumo estatístico
print("\n--- Resumo Estatístico ---")
print(df_train['nota_final_1000'].describe())

# 2. Vamos ver a contagem das notas mais comuns
print("\n--- Notas mais comuns (Top 20) ---")
print(df_train['nota_final_1000'].value_counts().head(20))

# 3. Vamos plotar um histograma para ver a distribuição visualmente
print("\n--- Gráfico da Distribuição ---")
plt.figure(figsize=(12, 6))
sns.histplot(df_train['nota_final_1000'], bins=20, kde=True)
plt.title('Distribuição das Notas Finais (0-1000) no Treino')
plt.xlabel('Nota Final')
plt.ylabel('Contagem (Número de Redações)')
plt.show()

SEU_TOKEN_AQUI

In [ ]:
# Célula 8: Instalar a biblioteca de integração do Hugging Face
!pip install huggingface_hub

In [ ]:
# Célula 9: Fazer Login no Hugging Face
from huggingface_hub import login

# Cole o seu Token (que começa com 'hf_...') que acabou de copiar
# É seguro, o 'input' esconde a sua palavra-passe
login(token = input("Cole o seu Token de 'write' do Hugging Face: "))

print("Login efetuado com sucesso!")

In [ ]:
# Célula 10: Forçar o Upload do CHECKPOINT-840 (O que encontrámos)

from huggingface_hub import HfApi, create_repo
from google.colab import drive
import os

# --- 1. Monte o seu Google Drive ---
print("A montar o Google Drive...")
drive.mount('/content/drive', force_remount=True)
print("Drive montado!")

# --- 2. Defina os Nomes ---

# --- A CORREÇÃO ESTÁ AQUI (baseado no que você encontrou) ---
# Vamos apontar para a pasta onde o ficheiro .safetensors ESTÁ
pasta_modelo_no_drive = "/content/drive/MyDrive/meu_modelo_bert_redacoes/checkpoint-840"

SEU_USERNAME_HF = "md43"
NOME_DO_MODELO_HF = "meu-bert-enem-v1"  # O nome do repositório (que já existe)
repo_id = f"{SEU_USERNAME_HF}/{NOME_DO_MODELO_HF}"

print(f"O modelo encontrado está em: {pasta_modelo_no_drive}")
print(f"O repositório de destino é: {repo_id}")

# --- 3. Verificação do Repositório (ignora se já existe) ---
try:
    create_repo(repo_id, private=False)
    print("Repositório novo criado.")
except Exception as e:
    print(f"Aviso: O repositório '{repo_id}' já existe. A carregar os ficheiros para a raiz...")
    pass

# --- 4. FAZER O UPLOAD (A parte importante) ---
api = HfApi()
print(f"A iniciar o UPLOAD dos ficheiros de '{pasta_modelo_no_drive}' para a RAIZ de '{repo_id}'...")
print("!!! ATENÇÃO: Isto é 1.4GB e pode demorar 5-10 minutos. NÃO INTERROMPA !!!")

# Esta função vai pegar em TUDO o que está dentro de 'checkpoint-840'
# e colocar na RAIZ do seu repositório HF.
api.upload_folder(
    folder_path=pasta_modelo_no_drive,
    repo_id=repo_id,
    repo_type="model"
)

print("\n--- SUCESSO! ---")
print(f"Os ficheiros do modelo (checkpoint-840) foram carregados para: https://huggingface.co/{repo_id}")

In [ ]:
# Célula 11: Carregar os Ficheiros do Dicionário (Tokenizer)

from transformers import AutoTokenizer
from huggingface_hub import HfApi, HfFolder
import os

# (Certifique-se que ainda está logado. Se der erro de permissão,
# execute a célula de login 'huggingface_hub.login()' primeiro)

# 1. Defina os nomes
NOME_BASE_MODELO = "neuralmind/bert-large-portuguese-cased" # O nosso modelo original
REPO_ID_DESTINO = "md43/meu-bert-enem-v1"                 # O seu repositório
PASTA_TEMP = "tokenizer_files_para_upload"              # Pasta temporária

print(f"A descarregar o tokenizer original de '{NOME_BASE_MODELO}'...")

# 2. Descarrega os ficheiros do tokenizer para uma pasta temporária
tokenizer = AutoTokenizer.from_pretrained(NOME_BASE_MODELO)
tokenizer.save_pretrained(PASTA_TEMP)

print(f"Ficheiros do tokenizer guardados em '{PASTA_TEMP}'. Os ficheiros são:")
print(os.listdir(PASTA_TEMP)) # Deve mostrar: vocab.txt, tokenizer_config.json, etc.

# 3. Carrega SÓ o tokenizer para o seu repositório
api = HfApi()
print(f"A carregar os ficheiros do tokenizer para a raiz de '{REPO_ID_DESTINO}'...")
print("Isto é rápido.")

api.upload_folder(
    folder_path=PASTA_TEMP,
    repo_id=REPO_ID_DESTINO,
    repo_type="model"
)

print("\n--- SUCESSO! ---")
print(f"O 'dicionário' (tokenizer) foi adicionado a: https://huggingface.co/{REPO_ID_DESTINO}")

In [ ]:
# ==============================================================================
# 0. CONFIGURAÇÃO (Execute esta célula)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuração visual dos gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

# Variáveis CONFIRMADAS do train_dataset_final.csv:
COLUNA_TEXTO_FINAL = 'texto_original'
COLUNA_NOTA_FINAL = 'nota_final_1000'
CAMINHO_DO_CSV = '/content/sample_data/train_dataset_final.csv'

# ==============================================================================
# 1. EXTRAÇÃO E PREPARAÇÃO DOS DADOS
# ==============================================================================

print("1. Carregando o CSV de treino e preparando os dados...")

try:
    # O separador ';' foi confirmado pelo seu script de ETL
    df = pd.read_csv(CAMINHO_DO_CSV, sep=';', encoding='utf-8-sig')

    # Validação e Limpeza
    if COLUNA_TEXTO_FINAL not in df.columns or COLUNA_NOTA_FINAL not in df.columns:
        raise ValueError("Colunas esperadas não foram encontradas. Verifique o CSV.")

    # --- CRIAÇÃO DE VARIÁVEL NUMÉRICA DERIVADA (NÃO FICTÍCIA) ---
    print(f"-> Dataset carregado com sucesso. Total de {len(df)} linhas.")
    print("-> Criando a variável 'tamanho_texto' (Comprimento da Redação)...")

    # Variável Numérica: Tamanho do Texto (Comprimento da Redação em caracteres)
    df['tamanho_texto'] = df[COLUNA_TEXTO_FINAL].astype(str).str.len()

    # Remove linhas onde a nota ou o texto é inválido
    df = df.dropna(subset=[COLUNA_NOTA_FINAL, 'tamanho_texto'])

    print(f"-> Dados prontos para análise Numérica. Total de {len(df)} linhas válidas.")

except FileNotFoundError:
    print(f"\nERRO CRÍTICO: Ficheiro '{CAMINHO_DO_CSV}' não encontrado.")
    print("Por favor, garanta que o arquivo está no diretório correto.")
    raise

# ==============================================================================
# 5.2.1. RELAÇÕES NUMÉRICA vs. NUMÉRICA (O ÚNICO REQUISITO CUMPRIDO)
# ==============================================================================

print("\n" + "="*70)
print("✅ 5.2.1. RELAÇÕES NUMÉRICA vs. NUMÉRICA (Nota vs. Tamanho do Texto)")
print("="*70)

variaveis_numericas = [COLUNA_NOTA_FINAL, 'tamanho_texto']

# 1. Cálculo da Matriz de Correlação
matriz_correlacao = df[variaveis_numericas].corr()
print("\n--- Matriz de Correlação de Pearson (Nota x Tamanho) ---")
print(matriz_correlacao.round(4))

# 2. Gráfico de Dispersão (Scatter Plot)
plt.figure(figsize=(9, 6))
sns.scatterplot(x='tamanho_texto', y=COLUNA_NOTA_FINAL, data=df, alpha=0.6, s=20)
plt.title(f'Relação entre Nota Final e Tamanho da Redação (em Caracteres)')
plt.xlabel('Tamanho do Texto (Nº de Caracteres)')
plt.ylabel('Nota Final (0-1000)')
plt.show()

# ==============================================================================
# 5.2.2. A 5.2.4. ANÁLISES CATEGÓRICAS E TEMPORAIS (NÃO REALIZADAS)
# ==============================================================================

print("\n" + "="*70)
print("❌ 5.2.2. a 5.2.4. ANÁLISES CATEGÓRICAS E TEMPORAIS")
print("="*70)
print("REQUISITOS NÃO CUMPRIDOS DEVIDO À ESTRUTURA DO DATASET (2 COLUNAS):")
print("* **Relações Categórica vs. Numérica:** Requer colunas categóricas (Tema, Corretor, Gênero).")
print("* **Relações Categórica vs. Categórica:** Requer pelo menos duas colunas categóricas.")
print("* **Análise Temporal:** Requer uma coluna de data ou timestamp.")
print("\nO dataset possui apenas dados de texto livre e uma nota, limitando a Análise Bivariada à relação entre Nota e Comprimento do Texto.")

In [ ]:
# ==============================================================================
# 0. CONFIGURAÇÃO (Execute esta célula)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuração visual dos gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

# Variáveis CONFIRMADAS do train_dataset_final.csv:
COLUNA_TEXTO_FINAL = 'texto_original'
COLUNA_NOTA_FINAL = 'nota_final_1000'
CAMINHO_DO_CSV = '/content/sample_data/train_dataset_final.csv'

# ==============================================================================
# 1. EXTRAÇÃO E PREPARAÇÃO DOS DADOS
# ==============================================================================

print("1. Carregando o CSV de treino e preparando os dados...")

try:
    # O separador ';' foi confirmado
    df = pd.read_csv(CAMINHO_DO_CSV, sep=';', encoding='utf-8-sig')

    # Validação e Limpeza
    if COLUNA_TEXTO_FINAL not in df.columns or COLUNA_NOTA_FINAL not in df.columns:
        raise ValueError("Colunas esperadas ('texto_original', 'nota_final_1000') não foram encontradas. Verifique o CSV.")

    # --- CRIAÇÃO DE VARIÁVEL NUMÉRICA DERIVADA ---
    print(f"-> Dataset carregado com sucesso. Total de {len(df)} linhas.")
    print("-> Criando a variável 'tamanho_texto' (Comprimento da Redação)...")

    # Variável Numérica: Tamanho do Texto (Comprimento da Redação em caracteres)
    df['tamanho_texto'] = df[COLUNA_TEXTO_FINAL].astype(str).str.len()

    # Remove linhas onde a nota ou o texto é inválido
    df = df.dropna(subset=[COLUNA_NOTA_FINAL, 'tamanho_texto'])

    # Garante que a nota é numérica para a correlação
    df[COLUNA_NOTA_FINAL] = pd.to_numeric(df[COLUNA_NOTA_FINAL], errors='coerce')
    df = df.dropna(subset=[COLUNA_NOTA_FINAL])

    print(f"-> Dados prontos para análise Numérica. Total de {len(df)} linhas válidas.")

except FileNotFoundError:
    print(f"\nERRO CRÍTICO: Ficheiro '{CAMINHO_DO_CSV}' não encontrado.")
    raise


# ==============================================================================
# 5.2.1. RELAÇÕES NUMÉRICA vs. NUMÉRICA (CUMPRINDO OS REQUISITOS)
# ==============================================================================

print("\n" + "="*70)
print("✅ 5.2.1. RELAÇÕES NUMÉRICA vs. NUMÉRICA (Nota vs. Tamanho do Texto)")
print("="*70)

variaveis_numericas = [COLUNA_NOTA_FINAL, 'tamanho_texto']

# 1. Cálculo da Matriz de Correlação
matriz_correlacao = df[variaveis_numericas].corr()
print("\n--- Matriz de Correlação de Pearson (Nota x Tamanho) ---")
print(matriz_correlacao.round(4))

# 2. Gráfico de Dispersão (Scatter Plot)
plt.figure(figsize=(9, 6))
sns.scatterplot(x='tamanho_texto', y=COLUNA_NOTA_FINAL, data=df, alpha=0.6, s=20)
plt.title(f'Relação entre Nota Final e Tamanho da Redação (em Caracteres)')
plt.xlabel('Tamanho do Texto (Nº de Caracteres)')
plt.ylabel('Nota Final (0-1000)')
plt.show()

# ==============================================================================
# AVISO SOBRE REQUISITOS NÃO CUMPRIDOS
# ==============================================================================

print("\n" + "="*70)
print("❌ AVISO: REQUISITOS CATEGÓRICOS/TEMPORAIS NÃO CUMPRIDOS")
print("="*70)
print("Os requisitos de 'Relações Categórica vs. Numérica', 'Relações Categórica vs. Categórica' e 'Análise Temporal' não podem ser gerados, pois o dataset contém apenas dados de Texto e a Nota Final.")